In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
!pip install requests pandas pyarrow tqdm

In [ ]:
#!/usr/bin/env python3
"""
funding_enrichment.py  —  Google Colab
======================================
Boost funding coverage the same way you boosted authorship: re-collect from
OpenAlex AND external sources using every join key in your metadata, then union.

WHY: OpenAlex `oa_has_funding` is FALSE on most rows and its funder metadata is
thin for Global-South work — the exact bias that would distort Paper 3's
funder-nationality network and Paper 1's funding layer. Crossref's Open Funder
Registry and Europe PMC's grant acknowledgements recover a large share of it.

THREE HARVESTERS (each independently resumable, cached to its own JSONL):
  1. OpenAlex grants   <- oa_id   (batched 50, like authorships; funder + award)
  2. Crossref funders  <- oa_doi  (Open Funder Registry: name + funder-DOI + award)
  3. Europe PMC grants <- oa_pmid or oa_doi (agency + grantId from acknowledgements)

FUNDER COUNTRY: built once from the OpenAlex /funders dump (~32k funders, each with
country_code + crossref-id + ror + name), then joined onto every harvested funder.
This is what makes the Paper-3 funder-NATIONALITY network possible.

OUTPUT COLUMNS (per oa_id):
  fund_oa            "NSF|US|award123; ..."      (OpenAlex grants)
  fund_crossref      "Wellcome Trust|GB|2020; ..."(Crossref)
  fund_epmc          "NIH|US|R01...; ..."         (Europe PMC)
  fund_funders_all   unique funder names (union across sources), ';'-joined
  fund_countries_all unique funder countries (union), ';'-joined
  fund_pairs         "name::country; ..."         (feed the bipartite network)
  fund_n_funders     int
  fund_has_funding   bool
  fund_sources       which sources yielded funding: "oa;crossref;epmc"

SETUP
  !pip install requests pandas pyarrow tqdm
  # add your email below for the polite pools (faster, more reliable)
"""

import json, re, sys, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import pandas as pd, requests
from google.colab import userdata

# =========================================================================== #
# CONFIG                                                                       #
# =========================================================================== #
IN_PARQUET  = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/llm_screen/screened_llm.parquet"
OUT_PARQUET = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich.parquet"
FUNDER_LOOKUP = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich/openalex_funders.parquet"
CACHE_OA   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich/fund_oa.jsonl"
CACHE_CR   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich/fund_crossref.jsonl"
CACHE_EP   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/funding_enrich/fund_epmc.jsonl"
MAILTO     = "cognosxx@gmail.com"


OPENALEX_KEY = userdata.get("OPENALEX_API_KEY")
# The line below was incorrectly overwriting the API key. It should be removed.
# OPENALEX_KEY = "api_key"

# only query the paid-effort external sources where OpenAlex found nothing?
EXTERNAL_ONLY_IF_MISSING = False      # True   # False = query Crossref/EPMC on all rows
CR_WORKERS = 6
EP_WORKERS = 6




def S(x):    # NA/None-safe string
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)

def short_id(x):
    return S(x).rsplit("/", 1)[-1].strip()

def norm_name(x):
    return re.sub(r"[^a-z0-9 ]", "", S(x).lower()).strip()

def norm_doi(x):
    d = S(x).lower().replace("https://doi.org/", "").replace("doi.org/", "").strip()
    return d or ""


# =========================================================================== #
# 1. FUNDER -> COUNTRY LOOKUP  (OpenAlex /funders dump, once)                  #
# =========================================================================== #
def build_funder_lookup(path=FUNDER_LOOKUP):
    if Path(path).exists():
        df = pd.read_parquet(path)
        print(f"[funders] loaded {len(df):,}", file=sys.stderr)
    else:
        # Create the parent directory if it doesn't exist
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        rows, cursor = [], "*"
        while cursor:
            r = requests.get("https://api.openalex.org/funders",
                             params={
                                 "per-page": 200,
                                 "cursor": cursor,
                                 "select": "id,display_name,country_code,ids",
                                 "mailto": MAILTO,
                                 "api_key": OPENALEX_KEY # Moved api_key into params dictionary
                             },
                             timeout=60)
            j = r.json()
            for f in j.get("results", []):
                ids = f.get("ids", {}) or {}
                rows.append({
                    "oa_funder": short_id(f.get("id")),
                    "name": f.get("display_name", ""),
                    "country": f.get("country_code", "") or "",
                    "crossref": short_id(ids.get("crossref", "")) if ids.get("crossref") else "",
                    "ror": short_id(ids.get("ror", "")) if ids.get("ror") else "",
                })
            cursor = j.get("meta", {}).get("next_cursor")
            print(f"[funders] {len(rows):,}", file=sys.stderr)
            time.sleep(0.1)
        df = pd.DataFrame(rows)
        df.to_parquet(path, index=False)
        print(f"[funders] wrote {path}", file=sys.stderr)

    # build fast lookups country by several keys
    by_oa, by_cr, by_ror, by_name = {}, {}, {}, {}
    for _, r in df.iterrows():
        if r["country"]:
            if r["oa_funder"]: by_oa[r["oa_funder"]] = r["country"]
            if r["crossref"]:  by_cr[r["crossref"]] = r["country"]
            if r["ror"]:       by_ror[r["ror"]] = r["country"]
            n = norm_name(r["name"])
            if n:              by_name.setdefault(n, r["country"])
    return {"oa": by_oa, "cr": by_cr, "ror": by_ror, "name": by_name}


# =========================================================================== #
# generic cache helpers                                                       #
# =========================================================================== #
def load_cache(path):
    done = {}
    if Path(path).exists():
        for line in open(path):
            try:
                d = json.loads(line); done[d["oa_id"]] = d
            except Exception:
                pass
    return done

def append_cache(fh, rec):
    fh.write(json.dumps(rec) + "\n"); fh.flush()


# =========================================================================== #
# 2. OPENALEX GRANTS  <- oa_id  (batched 50)                                   #
# =========================================================================== #
def harvest_openalex(ids):
    cache = load_cache(CACHE_OA)
    todo = [i for i in ids if i not in cache]
    print(f"[oa] {len(todo):,} to fetch", file=sys.stderr)
    fh = open(CACHE_OA, "a"); sess = requests.Session()
    for start in range(0, len(todo), 50):
        batch = todo[start:start+50]
        for attempt in range(5):
            try:
                r = sess.get("https://api.openalex.org/works",
                             params={
                                 "filter": f"ids.openalex:{'|'.join(batch)}",
                                 "select": "id,grants",
                                 "per-page": 50,
                                 "mailto": MAILTO,
                                 "api_key": OPENALEX_KEY # Moved api_key into params dictionary
                             },
                             timeout=60)
                if r.status_code == 200:
                    works = r.json().get("results", []); break
                if r.status_code in (429, 500, 502, 503):
                    time.sleep(2 ** attempt); continue
                r.raise_for_status()
            except requests.RequestException:
                time.sleep(2 ** attempt)
        else:
            works = []
        got = set()
        for w in works:
            oid = short_id(w.get("id")); got.add(oid)
            grants = [{"funder": short_id(g.get("funder")),
                       "name": g.get("funder_display_name", ""),
                       "award": g.get("award_id", "")}
                      for g in (w.get("grants") or [])]
            append_cache(fh, {"oa_id": oid, "grants": grants})
        for missing in set(batch) - got:
            append_cache(fh, {"oa_id": missing, "grants": []})
        if start % 2000 == 0:
            print(f"[oa] {start:,}/{len(todo):,}", file=sys.stderr)
        time.sleep(0.12)
    fh.close()
    return load_cache(CACHE_OA)


# =========================================================================== #
# 3. CROSSREF FUNDERS  <- oa_doi  (per-DOI, threaded)                          #
# =========================================================================== #
def _crossref_one(oa_id, doi, sess):
    for attempt in range(4):
        try:
            r = sess.get(f"https://api.crossref.org/works/{doi}",
                         params={"mailto": MAILTO}, timeout=45)
            if r.status_code == 404:
                return {"oa_id": oa_id, "funders": []}
            if r.status_code == 200:
                msg = r.json().get("message", {})
                fund = [{"name": f.get("name", ""),
                         "cr": short_id(f.get("DOI", "")) if f.get("DOI") else "",
                         "award": ";".join(f.get("award", []) or [])}
                        for f in (msg.get("funder") or [])]
                return {"oa_id": oa_id, "funders": fund}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"oa_id": oa_id, "funders": []}
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return {"oa_id": oa_id, "funders": []}

def harvest_crossref(df):
    cache = load_cache(CACHE_CR)
    sub = df[df["oa_doi"].map(lambda x: bool(norm_doi(x)))]
    todo = [(r["oa_id"], norm_doi(r["oa_doi"])) for _, r in sub.iterrows()
            if r["oa_id"] not in cache]
    print(f"[crossref] {len(todo):,} DOIs to query", file=sys.stderr)
    fh = open(CACHE_CR, "a")
    sess = requests.Session()
    sess.headers.update({"User-Agent": f"tick-sos-study/1.0 (mailto:{MAILTO})"})
    with ThreadPoolExecutor(max_workers=CR_WORKERS) as ex:
        futs = [ex.submit(_crossref_one, oid, doi, sess) for oid, doi in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            append_cache(fh, fut.result())
            if n % 500 == 0:
                print(f"[crossref] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return load_cache(CACHE_CR)


# =========================================================================== #
# 4. EUROPE PMC GRANTS  <- oa_pmid / oa_doi  (per-record, threaded)           #
# =========================================================================== #
def _epmc_one(oa_id, pmid, doi, sess):
    if pmid:
        query = f"EXT_ID:{pmid} AND SRC:MED"
    elif doi:
        query = f'DOI:"{doi}"'
    else:
        return {"oa_id": oa_id, "grants": []}
    for attempt in range(4):
        try:
            r = sess.get("https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                         params={"query": query, "resultType": "core",
                                 "format": "json", "pageSize": 1}, timeout=45)
            if r.status_code == 200:
                res = r.json().get("resultList", {}).get("result", [])
                if not res:
                    return {"oa_id": oa_id, "grants": []}
                grants = [{"agency": g.get("agency", ""), "grantId": g.get("grantId", "")}
                          for g in (res[0].get("grantsList", {}) or {}).get("grant", [])]
                return {"oa_id": oa_id, "grants": grants}
            if r.status_code in (429, 500, 502, 503):
                time.sleep(2 ** attempt); continue
            return {"oa_id": oa_id, "grants": []}
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return {"oa_id": oa_id, "grants": []}

def harvest_epmc(df):
    cache = load_cache(CACHE_EP)
    todo = [(r["oa_id"], S(r.get("oa_pmid")), norm_doi(r.get("oa_doi")))
            for _, r in df.iterrows()
            if r["oa_id"] not in cache and (S(r.get("oa_pmid")) or norm_doi(r.get("oa_doi")))]
    print(f"[epmc] {len(todo):,} to query", file=sys.stderr)
    fh = open(CACHE_EP, "a"); sess = requests.Session()
    with ThreadPoolExecutor(max_workers=EP_WORKERS) as ex:
        futs = [ex.submit(_epmc_one, oid, pmid, doi, sess) for oid, pmid, doi in todo]
        for n, fut in enumerate(as_completed(futs), 1):
            append_cache(fh, fut.result())
            if n % 500 == 0:
                print(f"[epmc] {n:,}/{len(todo):,}", file=sys.stderr)
    fh.close()
    return load_cache(CACHE_EP)


# =========================================================================== #
# 5. RECONCILE + attach country + write                                       #
# =========================================================================== #
def country_for(look, oa_funder="", cr="", name=""):
    if oa_funder and oa_funder in look["oa"]: return look["oa"][oa_funder]
    if cr and cr in look["cr"]:               return look["cr"][cr]
    n = norm_name(name)
    return look["name"].get(n, "")

def reconcile(df, look, oa_c, cr_c, ep_c):
    out = []
    for _, row in df.iterrows():
        oid = row["oa_id"]
        funders, countries, pairs, srcs = {}, set(), set(), set()
        parts_oa, parts_cr, parts_ep = [], [], []

        for g in (oa_c.get(oid, {}) or {}).get("grants", []):
            ctry = country_for(look, oa_funder=g["funder"], name=g["name"])
            nm = g["name"] or g["funder"]
            if nm: funders[norm_name(nm)] = nm; srcs.add("oa")
            if ctry: countries.add(ctry)
            if nm: pairs.add(f"{nm}::{ctry}")
            parts_oa.append(f"{nm}|{ctry}|{g['award']}")

        for f in (cr_c.get(oid, {}) or {}).get("funders", []):
            ctry = country_for(look, cr=f["cr"], name=f["name"])
            nm = f["name"]
            if nm: funders[norm_name(nm)] = nm; srcs.add("crossref")
            if ctry: countries.add(ctry)
            if nm: pairs.add(f"{nm}::{ctry}")
            parts_cr.append(f"{nm}|{ctry}|{f['award']}")

        for g in (ep_c.get(oid, {}) or {}).get("grants", []):
            ctry = country_for(look, name=g["agency"])
            nm = g["agency"]
            if nm: funders[norm_name(nm)] = nm; srcs.add("epmc")
            if ctry: countries.add(ctry)
            if nm: pairs.add(f"{nm}::{ctry}")
            parts_ep.append(f"{nm}|{ctry}|{g['grantId']}")

        names = sorted(set(funders.values()))
        out.append({
            "oa_id": oid,
            "fund_oa": "; ".join(parts_oa),
            "fund_crossref": "; ".join(parts_cr),
            "fund_epmc": "; ".join(parts_ep),
            "fund_funders_all": ";".join(names),
            "fund_countries_all": ";".join(sorted(countries)),
            "fund_pairs": "; ".join(sorted(pairs)),
            "fund_n_funders": len(names),
            "fund_has_funding": len(names) > 0,
            "fund_sources": ";".join(sorted(srcs)),
        })
    merged = df.merge(pd.DataFrame(out), on="oa_id", how="left")
    merged.to_parquet(OUT_PARQUET, index=False)

    # coverage report vs the old oa_has_funding
    new_cov = merged["fund_has_funding"].mean()
    if "oa_has_funding" in merged.columns:
        old = merged["oa_has_funding"].astype(str).str.upper().eq("TRUE").mean()
        print(f"[coverage] funding: {old:.1%} (OpenAlex only) -> {new_cov:.1%} (unioned)",
              file=sys.stderr)
    else:
        print(f"[coverage] funding now on {new_cov:.1%} of rows", file=sys.stderr)
    return merged


# =========================================================================== #
if __name__ == "__main__":
    df_all = pd.read_parquet(IN_PARQUET)
    df = df_all[df_all["llm_label"].notna()]
    ids = [short_id(x) for x in df["oa_id"].dropna().unique()]
    look = build_funder_lookup()

    oa_c = harvest_openalex(ids)

    if EXTERNAL_ONLY_IF_MISSING:
        has_oa = {k for k, v in oa_c.items() if (v or {}).get("grants")}
        ext_df = df[~df["oa_id"].isin(has_oa)]
        print(f"[external] querying {len(ext_df):,} rows lacking OpenAlex funding",
              file=sys.stderr)
    else:
        ext_df = df

    cr_c = harvest_crossref(ext_df)
    ep_c = harvest_epmc(ext_df)
    reconcile(df, look, oa_c, cr_c, ep_c)

In [ ]:
cr_c

In [ ]:
ep_c

In [ ]:
m =reconcile(df, look, oa_c, cr_c, ep_c)
m